# PeriClock 10-donor 完整示例

本 notebook 读取用户指定路径的外部 `sample_10donors.h5ad`，在内存中执行：Ensembl ID 保留与 gene symbol 映射、QC 指标、CP10K+log1p、CellTypist 注释、六个模型的特征构建和预测。不会写入原始 `.h5ad`、模型或结果文件。

样本有 152,472 个细胞。没有经过批准的本数据集 QC 阈值，因此本例只计算 QC 指标，**不按阈值删除细胞**。QC 与 CellTypist 按 donor 分块以限制内存；所有细胞均被处理。

See [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md) for resource identity, permissions and checksums. `PERICLOCK_RESOURCE_DIR` is optional common external storage; each existing file-path setting can override it. The example data and annotation model are not automatically downloaded.


Download the six clock models from [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock).
The [Dataset repository](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) is receiving uploads; check
[RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md) for per-file availability,
verified revisions and access instructions. Both repositories are currently private.
Configure actual external paths in the next cell. Historical outputs are cleared;
the full example has not been rerun during this release preparation. Full-reference
Symphony/BBKNN end-to-end validation remains incomplete. Selected paper analyses
are available in [Tutorials](../tutorials/README.md).


In [ ]:
from pathlib import Path
import gc, os, sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/periclock-mpl')
os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/periclock-numba')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/periclock-xdg')
import celltypist
import numpy as np
import pandas as pd
import scanpy as sc
import sklearn

examples_dir = Path.cwd().resolve()
release_root = next((p for p in [examples_dir, *examples_dir.parents] if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if release_root is None:
    raise RuntimeError("Start the example kernel from within the PeriClock code checkout")
resource_dir = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
# Edit these paths or set the corresponding environment variables.
input_path = Path(os.environ.get('PERICLOCK_EXAMPLE_DATA', str(resource_dir / 'datasets/sample_10donors.h5ad'))).expanduser()
celltypist_model_path = Path(os.environ.get('PERICLOCK_CELLTYPIST_MODEL', str(resource_dir / 'annotation/celltypist_model_cell_subtype_final.pkl'))).expanduser()
model_dir = Path(os.environ.get('PERICLOCK_MODEL_DIR', str(resource_dir / 'models'))).expanduser()
if not input_path.is_file():
    raise FileNotFoundError(f'Example data not found: {input_path}; download separately and set input_path.')
sys.path.insert(0, str(release_root))
from periclock import build_model_features, list_models, load_model, predict_age
from periclock.annotation import annotate_cells
from periclock.normalization import normalize_expression
from periclock.qc import calculate_qc_metrics

donor_key, age_key, annotation_key = 'donor_id', 'sample.subjectAgeAtDraw', 'cell_subtype_final'
adata = sc.read_h5ad(input_path)
print('Input:', adata.shape)
print('Donors:', adata.obs[donor_key].nunique())
print('Raw-like input:', adata.raw is None and not adata.layers)
print('Unique Ensembl IDs:', adata.var_names.is_unique)
print('Unique gene symbols:', adata.var['gene_symbols'].notna().all() and adata.var['gene_symbols'].is_unique)
print(f'scanpy {sc.__version__} | celltypist {celltypist.__version__} | scikit-learn {sklearn.__version__}')


## 1. Gene identifier mapping

CellTypist、线粒体 QC 和 release 模型均按 gene symbol 匹配。先将原始 Ensembl ID 保存到 `var['ensembl_id']`，仅替换内存对象的 `var_names`；不会修改输入文件。

In [ ]:
if 'gene_symbols' not in adata.var:
    raise KeyError("sample data has no var['gene_symbols'] mapping")
symbols = adata.var['gene_symbols'].astype(str)
if adata.var['gene_symbols'].isna().any() or not symbols.is_unique:
    raise ValueError('Gene symbols must be nonmissing and unique; do not silently aggregate duplicates.')
adata.var['ensembl_id'] = adata.var_names.astype(str)
adata.var_names = pd.Index(symbols, name='gene_symbols')
print("Mapped to unique gene symbols; original IDs retained in var['ensembl_id'].")


## 2. QC 指标与 normalization

`calculate_qc_metrics` 对每个 donor 副本计算 counts、detected genes 和 MT%。随后 `normalize_expression(..., copy=False)` 仅变更内存工作对象为 CP10K+log1p。

In [ ]:
donors = pd.Index(adata.obs[donor_key].astype(str).unique(), name=donor_key)
donor_values = adata.obs[donor_key].astype(str).to_numpy()
qc_rows = []
for donor in donors:
    positions = np.flatnonzero(donor_values == donor)
    donor_qc = calculate_qc_metrics(adata[positions].copy())
    qc_rows.append({donor_key: donor, 'n_cells': len(positions),
                    'median_genes': float(donor_qc.obs['n_genes_by_counts'].median()),
                    'median_pct_mt': float(donor_qc.obs['pct_counts_mt'].median())})
    del donor_qc
    gc.collect()
qc_summary = pd.DataFrame(qc_rows).set_index(donor_key).sort_index()
print(f'QC completed for {adata.n_obs} cells without threshold filtering.')
display(qc_summary.round(4))
adata = normalize_expression(adata, target_sum=1e4, copy=False)
print('Normalization:', adata.uns['periclock_normalization'])


## 3. CellTypist 注释

CellTypist 对单细胞独立分类，故按 donor 调用同一个 `annotate_cells(method='celltypist')` 接口并回填标签。`copy=False` 仅避免不必要的内存复制。

In [ ]:
labels = np.empty(adata.n_obs, dtype=object)
confidence = np.full(adata.n_obs, np.nan, dtype=float)
annotation_overlap = None
for donor in donors:
    positions = np.flatnonzero(donor_values == donor)
    annotated_donor = annotate_cells(adata[positions].copy(), method='celltypist', model_path=celltypist_model_path,
                                      label_key=annotation_key, copy=False)
    labels[positions] = annotated_donor.obs[annotation_key].astype(str).to_numpy()
    confidence[positions] = annotated_donor.obs['annotation_confidence'].to_numpy(dtype=float)
    annotation_overlap = annotated_donor.uns['annotation_gene_overlap']
    del annotated_donor
    gc.collect()
adata.obs[annotation_key] = labels
adata.obs['annotation_confidence'] = confidence
adata.uns['annotation_method'] = 'celltypist (donor-chunked example execution)'
adata.uns['annotation_gene_overlap'] = annotation_overlap
annotation_counts = adata.obs[annotation_key].value_counts()
print('CellTypist overlap:', annotation_overlap)
print('Annotated labels:', len(annotation_counts))
print('Top labels:', annotation_counts.head(10).to_dict())


## 3b. 其他 annotation 方法（按需运行）

下面两种方法也通过同一个 `periclock.annotation.annotate_cells` dispatcher 调用，但都需要额外的、带 `cell_subtype_final` 标签的 reference AnnData。它们不在本 notebook 默认执行：BBKNN+KNN 需要联合 reference/query PCA；Symphony 会建立 query-dependent reference，资源需求更高。请先确认 reference 的预处理状态、标签语义和数据使用权限。

In [ ]:
# BBKNN + KNN：reference 与 query 都应为同一表达尺度（通常是 log1p(CP10K)）。
# ref_path = Path('/approved/path/to/reference.h5ad')
# adata_bbknn = annotate_cells(
#     adata, method='bbknn_knn', reference=ref_path,
#     reference_label_key='cell_subtype_final', label_key='cell_subtype_final',
#     max_per_type=1000, n_hvg=2000, n_pcs=50,
#     neighbors_within_batch=10, k=20, confidence_threshold=0.5,
#     random_state=42, copy=True,
# )
# adata_bbknn.obs[['cell_subtype_final', 'annotation_confidence']].head()


In [ ]:
# Symphony：使用完整的原始 query-dependent Symphony reference-building 流程。
# ref_path = Path('/approved/path/to/reference.h5ad')
# adata_symphony = annotate_cells(
#     adata, method='symphony', reference=ref_path,
#     reference_label_key='cell_subtype_final', label_key='cell_subtype_final',
#     max_per_type=1000, n_hvg=2500, n_pcs=30, k=15,
#     random_state=42, copy=True,
# )
# adata_symphony.obs[['cell_subtype_final', 'annotation_confidence']].head()


## 4. 六个模型：特征构建与年龄预测

`build_model_features` 使用各 artifact 保存的特征顺序和转换器，`predict_age` 使用已训练 estimator。chronological age 只用于可选输出对照，未作为模型输入。以下结果是本固定输入、模型与依赖下的示例输出，不是历史训练一致性或临床性能声明。

In [ ]:
actual_age = pd.to_numeric(adata.obs.groupby(donor_key, observed=True)[age_key].first(), errors='raise')
predictions, feature_shapes = {}, {}
for model_name in list_models():
    bundle = load_model(model_name, model_dir=model_dir)
    X = build_model_features(adata, model_bundle=bundle, individual_key=donor_key,
                             celltype_key=annotation_key, max_zeros=28)
    feature_shapes[model_name] = X.shape
    predictions[model_name] = predict_age(X, model_name=model_name,
                                           model_bundle=bundle, actual_age=actual_age)
    print(f'{model_name}: feature matrix {X.shape}')
prediction_summary = pd.DataFrame({name: result['Predicted_Age']
                                   for name, result in predictions.items()})
prediction_summary.insert(0, 'Actual_Age', actual_age.reindex(prediction_summary.index))
prediction_summary.round(4)


## 5. 运行边界

本例未保存带注释/归一化的 AnnData，也没有重新训练、写入或移动模型。Bulk 缺失基因与 donor coverage 等科学规则仍遵循当前冻结 release；其未决科学含义见 `SCIENTIFIC_RULES_REVIEW.md`。